# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- **One row = one content item (page)**, built by grouping `fact_content_daily_performance`
  by `content_hash_id` over a single month, then joining `dim_content` for its metadata
  (word_count, content_type, main_intent). Same "one page" grain as ML-03, now sourced from
  the real warehouse instead of the pre-aggregated starter CSV.
- **Table(s):** `fact_content_daily_performance` (the `month=2026-03` partition — a full,
  mid-panel month, never the `_sample` table, which is the sealed final month / outcome
  window) + `dim_content` for metadata. `dim_clients` is touched only to read
  `gsc_data_start`/`ga4_data_start` for availability checks, never as a feature.
- **Time window:** `report_date` between 2026-03-01 and 2026-03-31 (31 days).
- **Predict/rank:** no supervised label — same as ML-03, the proxy is the cluster
  assignment itself, named only after inspecting centroids.
- **Deliberately excluded:** the hash-id keys (`client_hash_id`, `content_hash_id`,
  `keyword_hash_id`, `url_hash_id`). They're pseudonymized join/group keys with no ordinal
  meaning — using one as a feature would let the model memorize identity instead of learning
  from measured behavior.

In [1]:
import os
import duckdb

# HF_TOKEN lives in a local, gitignored .env — never paste a token into a cell (public repo!)
with open("../../.env") as f:
    for line in f:
        if line.startswith("HF_TOKEN"):
            os.environ["HF_TOKEN"] = line.strip().split("=", 1)[1]

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

MONTH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Section 1 claims, checked: row count, distinct content grain, and the date window
con.sql(f"""
    SELECT
        COUNT(*)                     AS n_rows,
        COUNT(DISTINCT content_hash_id) AS n_distinct_content,
        MIN(report_date)             AS min_date,
        MAX(report_date)             AS max_date
    FROM read_parquet('{MONTH}')
""").df()


,n_rows,n_distinct_content,min_date,max_date
0,9841378,331437,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature candidates** (aggregated per content item over the month, then joined to
  `dim_content`): `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (weighted),
  `ga4_engaged_sessions`, `ga4_sessions`, `scroll_events` — observed signals, all knowable
  at month-end. `word_count`, `content_type`, `main_intent` — static context features,
  knowable before the month even starts.
- **Label / proxy:** none supervised. Same as ML-03 — the proxy is the cluster assignment
  itself, named only after inspecting centroids.
- **Context (used, never as a feature):** `content_hash_id` / `client_hash_id` — grouping
  keys only. `dim_clients.gsc_data_start` / `ga4_data_start` — used only to filter for
  availability, never fed to the model.
- **Excluded:**
  - Hash-id keys (`client_hash_id`, `content_hash_id`, `keyword_hash_id`, `url_hash_id`) —
    pseudonyms with no ordinal meaning (see Section 1).
  - `ga4_*` columns for rows where `ga4_data_available = FALSE` — these are zero-FILLED
    placeholders, not real zero engagement. Averaging them in unfiltered mixes real
    behavior with padding, so they must be dropped with `WHERE ga4_data_available IS TRUE`
    before aggregating.

In [2]:
# Verify the ga4_data_available claim: how many rows are the zero-filled placeholder?
con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS n_rows,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM read_parquet('{MONTH}')
    GROUP BY ga4_data_available
""").df()


,ga4_data_available,n_rows,pct
0,False,6408671,65.1
1,<NA>,3018741,30.7
2,True,413966,4.2


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three checks below: (a) grain probe — after grouping by `content_hash_id`, no duplicates
survive; (b) row count + date span for the built frame (already confirmed 2026-03-01 →
2026-03-31 in Section 1); (c) availability — filtering `ga4_data_available IS TRUE` (which
correctly drops both the zero-filled `False` rows and the pre-`ga4_data_start` `NULL` rows,
per the investigation above), then a survivor count of how many content items still have
real GA4 data.

Then: the 5-feature frame (built from `month=2026-03`, each feature justified as knowable
at the decision moment), and the deliberate leakage trap — one label-derived column added,
the score jump shown, then removed.

In [3]:
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# (a) grain probe: dim_content itself must be one row per content item
grain_probe = con.sql(f"""
    SELECT content_hash_id, COUNT(*) AS n
    FROM read_parquet('{DIM_CONTENT}')
    GROUP BY content_hash_id
    HAVING COUNT(*) > 1
""").df()
print("dim_content duplicate content_hash_id rows:", len(grain_probe))  # expect 0

# (c) availability survivors: how many content items have ANY real GA4 row this month?
avail = con.sql(f"""
    SELECT
        COUNT(DISTINCT content_hash_id) AS n_content_total,
        COUNT(DISTINCT content_hash_id) FILTER (WHERE ga4_data_available IS TRUE)
            AS n_content_with_real_ga4
    FROM read_parquet('{MONTH}')
""").df()
print(avail.to_string())

# Build the 5-feature frame, grouped to the "one row = one content item" grain
feature_df = con.sql(f"""
    WITH gsc_agg AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks,
            SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0)
                AS avg_position
        FROM read_parquet('{MONTH}')
        GROUP BY content_hash_id
    ),
    ga4_agg AS (
        SELECT content_hash_id, SUM(ga4_engaged_sessions) AS engaged_sessions
        FROM read_parquet('{MONTH}')
        WHERE ga4_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        g.content_hash_id,
        g.total_impressions,
        g.total_clicks,
        g.avg_position,
        COALESCE(a.engaged_sessions, 0) AS engaged_sessions,
        dc.word_count
    FROM gsc_agg g
    LEFT JOIN ga4_agg a USING (content_hash_id)
    LEFT JOIN read_parquet('{DIM_CONTENT}') dc USING (content_hash_id)
""").df()

# (a) grain probe on the built frame: no content_hash_id should repeat after the joins
dupe_check = feature_df["content_hash_id"].duplicated().sum()
# (b) row count + window (window already proven in Section 1: 2026-03-01 to 2026-03-31)
print("feature_df rows:", len(feature_df), "| duplicate content_hash_id rows:", dupe_check)
feature_df.head()


dim_content duplicate content_hash_id rows: 0


   n_content_total  n_content_with_real_ga4
0           331437                    90489


feature_df rows: 331437 | duplicate content_hash_id rows: 0


,content_hash_id,total_impressions,total_clicks,avg_position,engaged_sessions,word_count
0,content_af187ecf59a5986d,1112.0,4.0,7.339928,0.0,3059
1,content_044eae5cec1e4ac1,72124.0,457.0,2.111641,27.0,2950
2,content_112bea2610d0a832,3541.0,16.0,13.995199,0.0,2592
3,content_0b616a8390a23b11,2416.0,1.0,10.731788,0.0,2801
4,content_89c194b7d806e4c8,812.0,1.0,4.934729,1.0,2929


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**GA4 engagement coverage is thin and unevenly distributed.** In `month=2026-03`, only
27.3% of content items (90,489 of 331,437) have any real GA4 row at all — the rest are
either zero-filled placeholders (`ga4_data_available = False`, client connected but flagged
unavailable) or genuinely absent (`ga4_data_available IS NULL`, the row predates the
client's `ga4_data_start` entirely). At the row level the split is even starker: 65.1%
`False`, 30.7% `NULL`, 4.2% `True`.

This means any archetype built on engagement behavior is only ever describing the ~27% of
the inventory with real GA4 history — not the whole panel. Clusters that lean on
`engaged_sessions` will systematically under-represent (or misrepresent, via the
zero-filled rows) newer clients and clients without a GA4 connection. This data can tell us
about on-site engagement for a biased subset, not for the inventory as a whole — a
limitation to carry into Section 4 of the capstone lane, not something Section 3's queries
can fix.

In [4]:
# Confirm the Section 4 claim: row-level and content-level GA4 coverage breakdown
row_level = con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS n_rows,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM read_parquet('{MONTH}')
    GROUP BY ga4_data_available
""").df()
print("Row-level ga4_data_available breakdown:")
print(row_level.to_string())

content_level_pct = 100 * avail["n_content_with_real_ga4"][0] / avail["n_content_total"][0]
print(f"\nContent-level coverage: {avail['n_content_with_real_ga4'][0]} / "
      f"{avail['n_content_total'][0]} = {content_level_pct:.1f}% of pages have any real GA4 data")


Row-level ga4_data_available breakdown:
   ga4_data_available   n_rows   pct
0               False  6408671  65.1
1                <NA>  3018741  30.7
2                True   413966   4.2

Content-level coverage: 90489 / 331437 = 27.3% of pages have any real GA4 data


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.